# JRC flood raster: interactive Folium viewer

Select an official JRC flood-depth GeoTIFF to inspect it interactively. This strict viewer draws only georeferenced native flood pixels: it never uses a preview-grid or raster-image overlay. Events above the configured pixel limit stop with an error instead of being approximated.

Run this notebook from the repository root with the project environment.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from flood_preview import build_strict_pixel_folium_map, discover_flood_raster_files, preview_summary, read_flood_preview

RASTER_ROOT = PROJECT_ROOT / 'data' / 'JRC_flood_depth_maps'
rasters = discover_flood_raster_files(RASTER_ROOT)
if not rasters:
    raise FileNotFoundError(f'No official JRC TIFF files found in {RASTER_ROOT}')

rasters_by_year = {}
for raster in rasters:
    rasters_by_year.setdefault(raster.year, []).append(raster)

print(f'Found {len(rasters):,} official JRC rasters in {RASTER_ROOT}')

In [ ]:
year_picker = widgets.Dropdown(
    options=sorted(rasters_by_year),
    description='Event year:',
    style={'description_width': 'initial'},
)

def options_for_year(year):
    return [
        (
            f'{item.start_date} to {item.end_date} | cluster {item.flood_id} | {item.raster_file}',
            str(item.path),
        )
        for item in rasters_by_year[year]
    ]

picker = widgets.Dropdown(
    options=options_for_year(year_picker.value),
    description='Flood raster:',
    layout=widgets.Layout(width='100%'),
    style={'description_width': 'initial'},
)

def update_raster_options(change):
    picker.options = options_for_year(change['new'])

year_picker.observe(update_raster_options, names='value')
MAX_NATIVE_PIXELS = 20_000
render_button = widgets.Button(description='Render exact-pixel map', button_style='primary', icon='map')
output = widgets.Output()

def render_map(_=None):
    with output:
        output.clear_output(wait=True)
        preview = read_flood_preview(
            picker.value,
            coarse_max_size=1200,
            detail_max_size=1800,
            threshold_cm=0.0,
            source_padding_pixels=600,
        )
        display(preview_summary(preview))
        display(build_strict_pixel_folium_map(preview, max_cells=MAX_NATIVE_PIXELS))

render_button.on_click(render_map)
display(widgets.VBox([year_picker, picker, render_button, output]))
render_map()

The displayed depths are in centimetres. This notebook deliberately has no approximate fallback: if an event exceeds `MAX_NATIVE_PIXELS`, it is not rendered. Raise that limit only when your browser can handle the corresponding number of polygons.